<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 4 (remaining hyperparameters)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 15 min).

Runs 2 and 3 tuned augmentation, dropout, learning rate and optimizer, frozen layers, and (Run 3) the decision threshold. The hyperparameter register in the proposal also marks **batch size**, **weight decay**, and **model scale (YOLOv8n-cls vs. YOLOv8s-cls)** as tuned, so this notebook tests each of them, one at a time, from the final configuration (R4: YOLOv8s-cls, dropout 0.3).

1. **Same data as Runs 2 and 3:** the Run 1 split minus the conflicting-label duplicates and the near-duplicate leak, rebuilt with the Run 2 code.
2. **Sweep on the validation set only (seed 42):** R4 (Run 2 final, re-run as the reference in this session), R8 batch 16, R9 weight decay 0, R10 weight decay 0.001, R11 YOLOv8n-cls.
3. **The held-out test set and the external set are not used.** They were already evaluated with the final configuration, so the final model does not change and these runs are reported as a sensitivity check on validation. A setting that helps here is noted in the report as future work.

Outputs go to `results/run4/` and are zipped as `run4_results.zip`.

In [ ]:
!pip -q install ultralytics==8.4.211 pi-heif imagehash kagglehub huggingface_hub

In [ ]:
import os, glob, shutil, time, json, random, zipfile, copy
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch, torchvision
import torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
EPOCHS = 30                  # max epochs per run; early stopping with patience 8 (hyperparameter register)
PATIENCE = 8
IMGSZ, BATCH = 224, 32
CNN_EPOCHS = 40              # scratch CNN baseline
FINAL_SEEDS = [42, 43, 44]   # the final configuration is trained with these seeds (mean and std)
NEAR_DUP_T = 5               # pHash Hamming distance at or below which two images count as near-duplicates
EFFNET_WEIGHTS = "IMAGENET1K_V1"
DATA_SRC = None              # set a folder path here to skip the Kaggle download
EXT_SRC = None               # set a folder or .zip path here to skip the Hugging Face download
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME if torch.cuda.is_available() else "none. Runtime > Change runtime type > T4 GPU, or this will take hours.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
ROOT = os.path.join(BASE, "coral_data")              # padded-square images in the Ultralytics folder layout
OUT = os.path.join(BASE, "results", "run4"); os.makedirs(OUT, exist_ok=True)
CLASSES = ["healthy", "bleached"]                    # index 1 = bleached, the positive class
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
T_START = time.time()

## 1. Data: the Run 2 split

Same code as Run 2 sections 1: the Run 1 split, minus every copy of the 9 photos filed under both labels, minus the validation or test copy of near-duplicates that cross splits.

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)
N_RAW = len(df)
print(N_RAW, "readable labeled images:", df["label"].value_counts().to_dict())

In [ ]:
splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
SAME_SPLIT = (int(df["split"].notna().sum()) == len(splits1)) and not df.duplicated(["file", "label"]).any()
print(f"Run 1 split matched: {SAME_SPLIT} ({int(df['split'].notna().sum())} of {len(splits1)} rows)")
if not SAME_SPLIT:
    print("WARNING: splits.csv did not match the download, so the split is rebuilt the Run 1 way (it may differ from Run 1).")
    from sklearn.model_selection import train_test_split
    keep = df.drop_duplicates("phash", keep="first")
    tr, tmp = train_test_split(keep, test_size=0.30, stratify=keep["label"], random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
    df["split"] = None
    df.loc[tr.index, "split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = sorted(n_lab[n_lab > 1].index)
conf = df[df["phash"].isin(conflict_hashes)].sort_values(["phash", "label"])
removed_conflict = conf[conf["split"].notna()]
print("Duplicate groups with conflicting labels:", len(conflict_hashes), f"({len(conf)} files)")
print("Copies removed from the Run 1 split:", removed_conflict["split"].value_counts().to_dict())

if len(conflict_hashes):
    groups = [g for _, g in conf.groupby("phash", sort=False)]
    per_row = 3; ncol = 2 * per_row; nrow = int(np.ceil(len(groups) / per_row))
    fig, axes = plt.subplots(nrow, ncol, figsize=(2.2 * ncol, 2.4 * nrow), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for g_i, g in enumerate(groups):
        for c_i, (_, r) in enumerate(g.head(2).iterrows()):
            ax = axes[g_i // per_row, (g_i % per_row) * 2 + c_i]
            ax.imshow(Image.open(r["path"]).convert("RGB")); ax.set_title(f"group {g_i + 1}: {r['label']}", fontsize=8)
    fig.suptitle("Same photo filed as both healthy and bleached (all copies removed)", fontsize=10)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "conflicting_duplicates.png"), dpi=150); plt.show()

In [ ]:
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def hamming(a, b):
    """Pairwise Hamming distances between two lists of 64-bit hex pHashes."""
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
iu = np.triu_indices(len(clean), k=1)
print("Image pairs by pHash distance:", {k: int((D[iu] == k).sum()) for k in range(0, 11)})
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
pairs = list(zip(ii, jj))
label_conflict = [(i, j) for i, j in pairs if lb[i] != lb[j]]
cross = [(i, j) for i, j in pairs if sp[i] != sp[j] and lb[i] == lb[j]]
rank = {"train": 0, "val": 1, "test": 2}
leak_drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in cross}       # keep the training copy
conflict_drop = {k for pair in label_conflict for k in pair}                 # label cannot be trusted
near_drop = sorted(leak_drop | conflict_drop)
print(f"Near-duplicate pairs (distance <= {NEAR_DUP_T}): {len(pairs)}; {len(cross)} same-label pairs across splits, "
      f"{len(label_conflict)} pairs with different labels")
print("Removed:", clean.loc[near_drop, "split"].value_counts().to_dict())

show = sorted(cross + label_conflict, key=lambda t: D[t])[:8]
if show:
    fig, axes = plt.subplots(len(show), 2, figsize=(4.6, 2.3 * len(show)), squeeze=False)
    for r_i, (i, j) in enumerate(show):
        for c_i, k in enumerate((i, j)):
            ax = axes[r_i, c_i]; ax.imshow(Image.open(clean.loc[k, "path"]).convert("RGB")); ax.axis("off")
            ax.set_title(f"{clean.loc[k, 'split']}, {clean.loc[k, 'label']} (d={D[i, j]})", fontsize=8)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "near_duplicates.png"), dpi=150); plt.show()

N_NEAR_PAIRS, N_CROSS, N_NEAR_CONFLICT = len(pairs), len(cross), len(label_conflict)
NEAR_REMOVED = clean.loc[near_drop, "split"].value_counts().to_dict()
clean = clean.drop(index=near_drop).reset_index(drop=True)
print(pd.crosstab(clean["split"], clean["label"], margins=True))
clean[["file", "label", "split", "phash"]].rename(columns={"file": "path"}).to_csv(os.path.join(OUT, "splits_v2.csv"), index=False)

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

shutil.rmtree(ROOT, ignore_errors=True)
for _, r in clean.iterrows():
    d = os.path.join(ROOT, r["split"], r["label"]); os.makedirs(d, exist_ok=True)
    pad_square(r["path"]).save(os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg"), quality=95)

def split_files(split, root=ROOT):
    files = sorted(glob.glob(os.path.join(root, split, "*", "*.jpg")))
    return files, np.array([CLASSES.index(os.path.basename(os.path.dirname(p))) for p in files])

(tr_files, y_tr), (va_files, y_va), (te_files, y_te) = (split_files(s) for s in ("train", "val", "test"))
assert len(tr_files) + len(va_files) + len(te_files) == len(clean), "some images were not written (file name clash)"
COUNTS = {s: {"n": len(y), "bleached": int(y.sum()), "healthy": int((y == 0).sum())}
          for s, y in (("train", y_tr), ("val", y_va), ("test", y_te))}
print("Data written to", ROOT, "| train/val/test:", len(tr_files), len(va_files), len(te_files))

In [ ]:
assert COUNTS == {"train": {"n": 629, "bleached": 334, "healthy": 295}, "val": {"n": 132, "bleached": 69, "healthy": 63},
                  "test": {"n": 131, "bleached": 67, "healthy": 64}}, f"split differs from Run 2: {COUNTS}"
print("Split matches Run 2:", COUNTS)

## 2. Metrics, the macro F1 callback, and the training helper

Same as Run 2, except that a stage can name the base weights with a `model` key (for YOLOv8n-cls) and the run folders go to `runs4/`.

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report
from ultralytics import YOLO
from ultralytics.utils.torch_utils import strip_optimizer

def metrics_from(y_true, p_bleached):
    """Metrics from P(bleached). The predicted class is bleached when P >= 0.5 (same as top-1 for two classes)."""
    p = np.asarray(p_bleached, dtype=float); y_pred = (p >= 0.5).astype(int)
    m = dict(macro_f1=f1_score(y_true, y_pred, average="macro"),
             bleached_recall=recall_score(y_true, y_pred, pos_label=1, zero_division=0),
             bleached_precision=precision_score(y_true, y_pred, pos_label=1, zero_division=0),
             healthy_recall=recall_score(y_true, y_pred, pos_label=0, zero_division=0),
             accuracy=accuracy_score(y_true, y_pred),
             roc_auc=roc_auc_score(y_true, p) if len(np.unique(p)) > 1 else 0.5)
    return m, y_pred

def track_macro_f1(model):
    """Log validation macro F1 and bleached recall each epoch; copy the checkpoint to best_f1.pt when macro F1 improves."""
    hist = []
    def on_fit_epoch_end(trainer):
        v = trainer.validator
        if v is None or not getattr(v, "training", False) or not v.pred:
            return  # skips the final re-validation of best.pt
        names = trainer.data["names"]
        names = names if isinstance(names, dict) else dict(enumerate(names))
        b = next(k for k, n in names.items() if n == "bleached")
        yp = (torch.cat(v.pred)[:, 0].numpy() == b).astype(int)
        yt = (torch.cat(v.targets).numpy() == b).astype(int)
        f1 = f1_score(yt, yp, average="macro")
        hist.append(dict(epoch=trainer.epoch + 1, macro_f1=f1, bleached_recall=recall_score(yt, yp, zero_division=0),
                         accuracy=float((yt == yp).mean())))
        if f1 > max((h["macro_f1"] for h in hist[:-1]), default=-1.0) and trainer.last.exists():
            shutil.copy(trainer.last, trainer.wdir / "best_f1.pt")
    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)
    return hist

def yolo_run(name, stages, seed=SEED):
    """Train one YOLOv8s-cls configuration. Each stage is a dict of Ultralytics overrides; stage 2 starts from stage 1's best."""
    weights, rows, best, t0 = stages[0].get("model", "yolov8s-cls.pt"), [], None, time.time()
    for k, over in enumerate(stages):
        args = dict(data=ROOT, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, deterministic=True, patience=PATIENCE,
                    project=os.path.join(BASE, "runs4"), name=f"{name}_s{seed}" + (f"_st{k + 1}" if len(stages) > 1 else ""),
                    exist_ok=True, device=DEVICE, plots=False, verbose=False)
        args.update({k: v for k, v in over.items() if k != "model"})
        model = YOLO(weights)
        hist = track_macro_f1(model)
        model.train(**args)
        weights = str(model.trainer.wdir / "best_f1.pt")
        for h in hist:
            h["stage"] = k + 1
            if best is None or h["macro_f1"] > best["macro_f1"]:
                best = dict(h, weights=weights)
        rows += hist
    strip_optimizer(best["weights"])
    h = pd.DataFrame(rows); h["step"] = np.arange(1, len(h) + 1)
    settings = " | ".join(", ".join(f"{a}={b}" for a, b in s.items()) or "Run 1 settings" for s in stages)
    return dict(name=name, seed=seed, stages=stages, settings=settings, best_epoch=int(best["epoch"]), best_stage=int(best["stage"]),
                macro_f1=best["macro_f1"], bleached_recall=best["bleached_recall"], accuracy=best["accuracy"],
                epochs_run=len(h), minutes=(time.time() - t0) / 60, weights=best["weights"], hist=h)

def predict_yolo(weights, files, bs=64):
    m = YOLO(weights)
    b = next(k for k, n in m.names.items() if n == "bleached")
    p = []
    for k in range(0, len(files), bs):
        p += [float(r.probs.data[b]) for r in m.predict(files[k:k + bs], imgsz=IMGSZ, device=DEVICE, verbose=False)]
    return np.array(p)

## 3. Sweep: one setting at a time from the Run 2 final configuration

| Run | Change from R4 (dropout 0.3) | Why (hyperparameter register) |
| :-- | :-- | :-- |
| R4 | none (the final configuration, re-run here as the reference) | same session and GPU, so the comparison is fair |
| R8 | batch size 16 (from 32) | smaller batches add gradient noise, which can regularize on about 630 images |
| R9 | weight decay 0 (from 0.0005, the Ultralytics default) | checks whether L2 regularization helps at all |
| R10 | weight decay 0.001 | stronger L2 regularization against overfitting |
| R11 | YOLOv8n-cls (about 1.4M parameters with the 2-class head, from 5.1M) | a smaller model may overfit less and is faster on a CPU |

All runs: seed 42, up to 30 epochs, patience 8, best-macro-F1 checkpoint, validation set only.

In [ ]:
SWEEP3 = {
    "R4_dropout0.3": [dict(dropout=0.3)],
    "R8_batch16": [dict(dropout=0.3, batch=16)],
    "R9_wd0": [dict(dropout=0.3, weight_decay=0.0)],
    "R10_wd1e-3": [dict(dropout=0.3, weight_decay=1e-3)],
    "R11_yolov8n": [dict(model="yolov8n-cls.pt", dropout=0.3)],
}
results3 = []
for name, stages in SWEEP3.items():
    r = yolo_run(name, stages)
    r["params_M"] = sum(p.numel() for p in YOLO(r["weights"]).model.parameters()) / 1e6
    r["size_MB"] = os.path.getsize(r["weights"]) / 1e6
    results3.append(r)
    print(f"\n>>> {name}: val macro F1 {r['macro_f1']:.3f}, bleached recall {r['bleached_recall']:.3f} "
          f"(epoch {r['best_epoch']}, {r['epochs_run']} epochs run, {r['minutes']:.1f} min, {r['params_M']:.1f}M params)\n")

sweep3 = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "stages")} for r in results3])
ref3 = sweep3.loc[sweep3["name"] == "R4_dropout0.3", "macro_f1"].iloc[0]
sweep3["delta_vs_R4"] = sweep3["macro_f1"] - ref3
sweep3.to_csv(os.path.join(OUT, "sweep4_results.csv"), index=False)
pd.concat([r["hist"].assign(run=r["name"]) for r in results3]).to_csv(os.path.join(OUT, "sweep4_epoch_history.csv"), index=False)
print(sweep3[["name", "macro_f1", "delta_vs_R4", "bleached_recall", "accuracy", "best_epoch", "epochs_run", "params_M", "size_MB", "minutes"]]
      .round(3).to_string(index=False))

## 4. Learning curves for each tuned setting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(sweep3["name"])
axes[0].bar(names, sweep3["macro_f1"], color=COLORS[:len(names)])
for i, v in enumerate(sweep3["macro_f1"]):
    axes[0].text(i, v + 0.004, f"{v:.3f}", ha="center", fontsize=9)
axes[0].set_ylim(max(0, sweep3["macro_f1"].min() - 0.08), min(1, sweep3["macro_f1"].max() + 0.05))
axes[0].set_ylabel("Validation macro F1 (best epoch)"); axes[0].set_title("Run 4 sweep, seed 42")
axes[0].tick_params(axis="x", rotation=20)
for i, r in enumerate(results3):
    axes[1].plot(r["hist"]["step"], r["hist"]["macro_f1"], marker="o", ms=3, label=r["name"], color=COLORS[i])
axes[1].axhline(0.90, ls="--", color="gray", lw=1); axes[1].text(1, 0.902, "0.90 target", color="gray", fontsize=8)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation macro F1"); axes[1].set_title("Validation macro F1 per epoch")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "sweep4_macro_f1.png"), dpi=150); plt.show()

## 5. Summary for the progress log

In [ ]:
lines = [f"RUN 4 SUMMARY ({time.strftime('%b. %d, %Y')})",
         "Data: Run 2 split (train/val/test " + "/".join(str(COUNTS[s]["n"]) for s in ("train", "val", "test")) + "); test set not used.",
         "Sweep from the final configuration (R4: YOLOv8s-cls, dropout 0.3), seed 42, validation set:"]
for _, r in sweep3.iterrows():
    lines.append(f"  {r['name']}: macro F1 {r['macro_f1']:.3f} ({r['delta_vs_R4']:+.3f} vs R4), bleached recall {r['bleached_recall']:.3f}, "
                 f"best epoch {r['best_epoch']} of {r['epochs_run']}, {r['params_M']:.1f}M params, {r['size_MB']:.1f} MB [{r['settings']}]")
best3 = sweep3.loc[sweep3["macro_f1"].idxmax()]
lines.append(f"Best on validation: {best3['name']} ({best3['macro_f1']:.3f}). The final model does not change because the test set was already used.")
lines.append(f"Environment: GPU {GPU_NAME}, ultralytics {__import__('ultralytics').__version__}, torch {torch.__version__}. "
             f"Total time {(time.time() - T_START) / 60:.1f} min.")
summary = "\n".join(lines); print(summary)
open(os.path.join(OUT, "run4_summary.txt"), "w").write(summary)

zp = os.path.join(BASE, "run4_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(OUT, "*"))):
        z.write(f, os.path.basename(f))
print("\nZipped:", zp)
try:
    from google.colab import files as colab_files
    colab_files.download(zp)
except ImportError:
    pass